# Extension du tokenizer wolof pour SmolLM3-3B-Base

---

## Ce que fait ce notebook

| Phase | Contenu | Livrable |
|---|---|---|
| 0 | Environnement et vérifications | — |
| 1 | Corpus et splits figés | `wolof_eval_v1.parquet`, `wolof_bpe_train_v1.parquet` |
| 2 | Fertilité de référence | chiffre baseline |
| 3 | Construction du tokenizer étendu | `tokenizers/smollm3_wolof_ext_8000/` |
| 4 | Validations obligatoires | monotonicité, round-trip, non-régression |
| 5 | Chirurgie des embeddings | `models/smollm3-wolof-ext8000-init/` |
| 6 | BPB de référence | métrique de comparaison |

## Vocabulaire

- **Fertilité** = nombre moyen de tokens par mot. Plus c'est bas, plus le tokenizer est efficace sur la langue.
- **BPB** (*bits-per-byte*) = information par octet de texte brut. **Seule métrique valide** pour comparer deux modèles ayant des tokenizers différents, car elle ne dépend pas du découpage.
- **Mean-pooling** = initialiser un nouveau token par la moyenne des embeddings de ses anciens sous-tokens.

## Phase 0  Environnement

In [ ]:
# transformers >= 4.53.0 est la version qui intègre le code de modélisation de SmolLM3.
!pip install -q "transformers>=4.53.0" datasets pandas pyarrow

In [ ]:
import transformers, pandas as pd
print("transformers :", transformers.__version__)

In [ ]:
ROOT        = "/content/drive/MyDrive/SmolLM3-3B-Base"
BASE_ID     = "HuggingFaceTB/SmolLM3-3B-Base"
CORPUS_PATH = f"{ROOT}/data/wolof_corpus_clean_token_v1.parquet"
EVAL_SIZE   = 5000
SEED        = 42
N_FINAL     = 8000          # taille d'extension retenue (voir Phase 3)

### Piège n1  `vocab_size` ≠ `len(tokenizer)`

SmolLM3 annonce `vocab_size = 128000`, mais le tokenizer contient réellement **128 256** entrées :
128 000 tokens BPE (ids 0 → 127 999) **plus** 256 tokens spéciaux (ids 128 000 → 128 255).

Dimensionner la matrice d'embeddings sur `vocab_size` produit des indices hors bornes,
et l'erreur n'apparaît qu'en pleine phase d'entraînement.

**Toujours utiliser `len(tokenizer)`.**

In [ ]:
from transformers import AutoTokenizer
base_tok = AutoTokenizer.from_pretrained(BASE_ID)

print("vocab_size      :", base_tok.vocab_size)      # 128000
print("len(tokenizer)  :", len(base_tok))            # 128256  <-- LA bonne valeur
print("écart           :", len(base_tok) - base_tok.vocab_size)
print("type            :", base_tok.__class__.__name__)

## Phase 1 Corpus et splits figés

Le split d'évaluation doit être **identique pour tous les runs** de la grille d'ablation,
sinon les comparaisons ne valent rien. La graine (`SEED`) le garantit, et les splits sont
sauvegardés sur disque pour ne jamais être régénérés.

In [ ]:
df = pd.read_parquet(CORPUS_PATH)
print("lignes :", len(df), "| colonnes :", df.columns.tolist())

shuffled = df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
eval_df  = shuffled.iloc[:EVAL_SIZE].reset_index(drop=True)
train_df = shuffled.iloc[EVAL_SIZE:].reset_index(drop=True)
print(f"éval : {len(eval_df):,} | train : {len(train_df):,}")

In [ ]:
import os
os.makedirs(f"{ROOT}/data", exist_ok=True)
eval_df.to_parquet(f"{ROOT}/data/wolof_eval_v1.parquet", index=False)
train_df.to_parquet(f"{ROOT}/data/wolof_bpe_train_v1.parquet", index=False)
print("splits figés")

### Contrôle de fuite train / éval

Le corpus source n'a subi qu'une déduplication **exacte** : des quasi-doublons
(casse, ponctuation, espaces) peuvent subsister. Comme le split est tiré du même corpus,
une fuite gonflerait artificiellement tous les résultats.

In [ ]:
import re

def normaliser(t):
    t = str(t).lower()
    t = re.sub(r"[^\w\s]", " ", t)
    return re.sub(r"\s+", " ", t).strip()

train_norm = set(train_df["text"].map(normaliser))
fuite = eval_df["text"].map(normaliser).isin(train_norm)
print(f"quasi-doublons éval↔train : {fuite.sum()} / {len(eval_df)}  ({100*fuite.mean():.2f} %)")

## Phase 2  Fertilité de référence

La fertilité se mesure **en contexte** (phrases entières, espaces compris).
Mesurer sur des mots isolés donne des chiffres trompeurs, car un BPE *byte-level*
encode différemment un mot précédé d'un espace.

In [ ]:
def compute_fertility(tokenizer, texts):
    nw = nt = nc = 0
    for t in texts:
        t = "" if t is None else str(t)
        w = len(t.split())
        if w == 0: continue
        ids = tokenizer.encode(t, add_special_tokens=False)
        nw += w; nt += len(ids); nc += len(t)
    return {"tokens_per_word": nt/max(nw,1),
            "tokens_per_char": nt/max(nc,1),
            "n_words": nw, "n_tokens": nt}

eval_texts = eval_df["text"].tolist()
r = compute_fertility(base_tok, eval_texts)
FERT_BASE = r["tokens_per_word"]
print(f"FERTILITÉ BASELINE : {FERT_BASE:.3f} tokens/mot")
print(f"tokens/caractère   : {r['tokens_per_char']:.3f}")
print(f"vocabulaire        : {len(base_tok):,}")

### Voir la fragmentation

SmolLM3 n'ayant jamais vu de wolof, les lettres accentuées (ë, ñ, ó, à) sont éclatées
au niveau des **octets UTF-8**. La lettre `ñ` coûte à elle seule 2 tokens.

In [ ]:
MOTS_TEST = ["jëfandikoo","ñëw","wóoral","càkuteef","jëwriñu","réew","waaw","liggéeyam","ñu","dafay"]
for m in MOTS_TEST:
    ids = base_tok.encode(" " + m, add_special_tokens=False)   # espace initial = usage réel
    print(f"{m:12} -> {len(ids)} tokens : {base_tok.convert_ids_to_tokens(ids)}")

## Phase 3  Construction du tokenizer étendu

### Découverte déterminante : `ignore_merges = True`

Le tokenizer de SmolLM3 a le paramètre `ignore_merges` à **`True`**.
Cela signifie qu'avant d'appliquer la moindre règle de fusion, le tokenizer
**cherche le mot entier dans le vocabulaire** ; s'il l'y trouve, il le sort en un seul token.

Conséquences directes sur la stratégie :

1. **Il suffit d'ajouter des formes de mots entiers au vocabulaire**  sans toucher aux merges
   pour obtenir 1 token par mot.
2. **La monotonicité est garantie par construction** : un mot ajouté devient 1 token ;
   un mot absent suit exactement le chemin d'avant, puisque les merges n'ont pas bougé.
   Aucune régression n'est possible.
3. **Ajouter des sous-mots serait inutile** : un sous-mot n'est atteignable que par les merges ;
   sans eux, il resterait une ligne d'embedding morte.

C'est pourquoi la sélection ci-dessous se fait par **fréquence de mots entiers**
et non à partir du vocabulaire d'un BPE auxiliaire.

In [ ]:
import json
raw = json.loads(base_tok.backend_tokenizer.to_str())
print("type modèle    :", raw["model"]["type"])
print("ignore_merges  :", raw["model"].get("ignore_merges"))     # True
print("nb merges      :", len(raw["model"]["merges"]))
print("format merges  :", type(raw["model"]["merges"][0]).__name__)

### Étape 3.1  Compter les mots

On utilise le **pré-tokenizer de SmolLM3 lui-même**, pour obtenir exactement les segments
qu'il produira à l'inférence (déjà convertis en surface *byte-level*).

In [ ]:
from collections import Counter

pre = base_tok.backend_tokenizer.pre_tokenizer
N_SAMPLE = 300_000

sample = train_df["text"].astype(str).sample(
    n=min(N_SAMPLE, len(train_df)), random_state=SEED).tolist()

cnt = Counter()
for t in sample:
    for chunk, _ in pre.pre_tokenize_str(t):
        cnt[chunk] += 1

print("segments distincts :", f"{len(cnt):,}")
print("exemples :", cnt.most_common(8))

### Étape 3.2  Sélectionner les candidats

Le tri se fait par **gain de tokens attendu** = `fréquence × (nb_tokens − 1)`,
c'est-à-dire le nombre de tokens réellement économisés sur le corpus.
On ajoute donc en priorité ce qui fait le plus baisser la fertilité,
et non simplement les mots les plus fréquents.

In [ ]:
base_vocab = set(base_tok.get_vocab())

def surface_to_text(s):
    return base_tok.backend_tokenizer.decoder.decode([s])

candidats = []
for chunk, freq in cnt.most_common():
    if chunk in base_vocab:            # déjà 1 token, rien à gagner
        continue
    n_tok = len(base_tok.encode(surface_to_text(chunk), add_special_tokens=False))
    if n_tok < 2:                      # déjà efficace
        continue
    txt = surface_to_text(chunk).strip()
    if len(txt) < 2 or not any(c.isalpha() for c in txt):
        continue
    candidats.append((chunk, freq, n_tok, freq * (n_tok - 1)))

candidats.sort(key=lambda x: -x[3])
print("candidats :", f"{len(candidats):,}")
for c, f, n, g in candidats[:10]:
    print(f"  {c!r:20} freq={f:7,}  {n} tokens → 1   gain≈{g:,}")

### Étape 3.3  Construire le tokenizer étendu

#### Piège n°2  la réindexation des identifiants

SmolLM3 range ses tokens ainsi : BPE aux ids `0 → 127 999`, spéciaux aux ids `128 000 → 128 255`.

Injecter naïvement de nouveaux tokens dans `model.vocab` crée **soit une collision**
avec les tokens spéciaux, **soit un trou** dans la numérotation — et la bibliothèque
`tokenizers` ne supporte ni l'un ni l'autre. Symptôme observé : des mots wolof décodés
en `<|reserved_special_token_N|>`.

**Solution : réindexer.** Les nouveaux tokens sont placés de façon **contiguë** juste après
le vocabulaire BPE (`V → V+N-1`), et les 256 tokens spéciaux sont **décalés au-dessus**
(`V+N → V+N+255`).

Le `id_map` retourné est **indispensable** pour la Phase 5 : sans lui, impossible de déplacer
correctement les lignes d'embedding des tokens spéciaux.

In [ ]:
import json, os, tempfile
from transformers import AutoTokenizer

def build_extended(n_add):
    tok = AutoTokenizer.from_pretrained(BASE_ID)
    j = json.loads(tok.backend_tokenizer.to_str())
    v = j["model"]["vocab"]
    V = max(v.values()) + 1                       # taille du vocab BPE = 128000
    specials = sorted(j.get("added_tokens", []), key=lambda a: a["id"])

    nouveaux = [c for c, _, _, _ in candidats[:n_add] if c not in v]
    N = len(nouveaux)

    # 1) nouveaux tokens CONTIGUS juste après le vocab BPE
    for i, s in enumerate(nouveaux):
        v[s] = V + i

    # 2) tokens spéciaux décalés au-dessus (on trace le remapping)
    id_map = {}
    for k, a in enumerate(specials):
        id_map[a["id"]] = V + N + k
        a["id"] = V + N + k

    j["model"]["vocab"] = v
    j["added_tokens"]   = specials

    d = tempfile.mkdtemp(); tok.save_pretrained(d)
    with open(os.path.join(d, "tokenizer.json"), "w", encoding="utf-8") as f:
        json.dump(j, f, ensure_ascii=False)
    for fn in ("vocab.json", "merges.txt"):       # éviter que le reload lise les fichiers legacy
        p = os.path.join(d, fn)
        if os.path.exists(p): os.remove(p)

    return AutoTokenizer.from_pretrained(d), {"V": V, "N": N, "id_map": id_map}

### Étape 3.4 Choisir la taille d'extension

Le choix se fait au **coude de la courbe**, en tenant compte d'un arbitrage souvent négligé :

> **fertilité contre entraînabilité**

Chaque token ajouté est une ligne d'embedding qu'il faudra **entraîner**. Un token trop rare
recevra très peu de gradients et restera proche de son initialisation un mot découpé en
sous-mots bien entraînés peut alors être mieux modélisé qu'un token unique mal entraîné.

La colonne `freq. dernier` sert donc autant que la colonne `fertilité` pour décider.

In [ ]:
print(f"{'ajoutés':>9} {'fertilité':>10} {'freq. dernier':>15}")
print(f"{0:>9} {FERT_BASE:>10.3f} {'—':>15}")
for n in [1000, 2000, 4000, 8000, 16000]:
    e, _ = build_extended(n)
    f = compute_fertility(e, eval_texts)["tokens_per_word"]
    print(f"{n:>9} {f:>10.3f} {candidats[n-1][1]:>15,}")

In [ ]:
ext_tok, meta = build_extended(N_FINAL)
V, N   = meta["V"], meta["N"]
id_map = meta["id_map"]
FERT_EXT = compute_fertility(ext_tok, eval_texts)["tokens_per_word"]

print(f"V={V} | ajoutés={N:,} | vocabulaire : {len(base_tok):,} → {len(ext_tok):,}")
print(f"fertilité : {FERT_BASE:.3f} → {FERT_EXT:.3f}  ({100*(1-FERT_EXT/FERT_BASE):.1f} % de gain)")

## Phase 4 Les trois validations obligatoires

| Test | Ce qu'il vérifie | Attendu |
|---|---|---|
| **Monotonicité** | aucun mot n'est dégradé par l'extension | `pire = 0` |
| **Round-trip** | encoder puis décoder rend le texte identique | même score que la base |
| **Non-régression FR/EN** | le français et l'anglais sont intacts | `identique` partout |

Le troisième test est **crucial pour les runs multilingues** : si l'extension dégradait
le français ou l'anglais, elle fausserait ces expériences.

In [ ]:
# 1) MONOTONICITÉ — doit être 0 par construction
mots = []
for t in eval_texts:
    for w in str(t).split():
        c = w.strip(".,;:!?\"'()[]{}«»—–…·0123456789")
        if c and c.isalpha(): mots.append(c)

pire = mieux = 0
for c in mots[:20000]:
    d = len(base_tok.encode(" "+c, add_special_tokens=False))
    e = len(ext_tok.encode(" "+c, add_special_tokens=False))
    if e > d: pire += 1
    elif e < d: mieux += 1
print(f"MONOTONICITÉ  pire={pire}  mieux={mieux}  → {'OK' if pire==0 else 'PROBLÈME'}")

Le round-trip se mesure avec `clean_up_tokenization_spaces=False`.
Avec `True`, le décodeur de SmolLM3 retouche les espaces autour de la ponctuation
et fait échouer le test **même sur le tokenizer de base** — c'est un artefact du décodeur,
pas un défaut de l'extension. On compare donc toujours au tokenizer de base.

In [ ]:
def roundtrip(tok, textes, clean=False):
    return sum(tok.decode(tok.encode(t, add_special_tokens=False),
                          clean_up_tokenization_spaces=clean) == t for t in textes)

ech = eval_texts[:500]
print(f"ROUND-TRIP (clean=False)  base : {roundtrip(base_tok, ech)}/500 "
      f"| étendu : {roundtrip(ext_tok, ech)}/500")

In [ ]:
# 3) NON-RÉGRESSION FR / EN / code
for phrase in ["Le chat dort sur le canapé.",
               "The quick brown fox jumps over the lazy dog.",
               "def add(a, b): return a + b"]:
    a = base_tok.encode(phrase, add_special_tokens=False)
    b = ext_tok.encode(phrase, add_special_tokens=False)
    print(f"  {'identique' if a==b else 'DIFFÉRENT'} ({len(a)}→{len(b)}) : {phrase[:42]}")

### Figer le tokenizer

Le tokenizer est **construit une seule fois puis chargé depuis le disque** dans tous les runs.
Le régénérer à chaque entraînement ne garantirait pas qu'il soit strictement identique
(l'échantillon de comptage et l'ordre de parcours peuvent varier) deux runs « étendus »
reposeraient alors sur deux tokenizers différents, sans que rien ne l'indique.

In [ ]:
TOK_DIR = f"{ROOT}/tokenizers/smollm3_wolof_ext_{N_FINAL}"
os.makedirs(TOK_DIR, exist_ok=True)
ext_tok.save_pretrained(TOK_DIR)

with open(f"{TOK_DIR}/extension_meta.json", "w", encoding="utf-8") as f:
    json.dump({
        "base_model": BASE_ID,
        "methode": "mots entiers par frequence (ignore_merges=True), reindexation des ids",
        "n_added": N,
        "vocab_bpe_size": V,
        "vocab_total_before": len(base_tok),
        "vocab_total_after": len(ext_tok),
        "id_map_specials": {str(k): v for k, v in id_map.items()},
        "fertility_default": FERT_BASE,
        "fertility_extended": FERT_EXT,
        "eval_split": "data/wolof_eval_v1.parquet",
        "seed": SEED,
    }, f, indent=2, ensure_ascii=False)

print("tokenizer figé :", TOK_DIR)

## Phase 5 Chirurgie des embeddings

### La carte des matrices

| | Avant (128 256 lignes) | Après (`len(ext_tok)` lignes) |
|---|---|---|
| BPE | `0 → V-1` | `0 → V-1`  **inchangés** |
| Nouveaux wolof | | `V → V+N-1`  **mean-pooling** |
| Spéciaux | `V → V+255` | `V+N → V+N+255`  **déplacés** |

Trois opérations dans cet ordre : agrandir, **déplacer** les lignes spéciales
(sinon `eos` pointerait vers un mot wolof), puis **initialiser** les nouvelles.

In [ ]:
import torch
from transformers import AutoModelForCausalLM

ext_tok  = AutoTokenizer.from_pretrained(TOK_DIR)
meta     = json.load(open(f"{TOK_DIR}/extension_meta.json"))
V, N     = meta["vocab_bpe_size"], meta["n_added"]
id_map   = {int(k): v for k, v in meta["id_map_specials"].items()}
assert len(ext_tok) == meta["vocab_total_after"], "tokenizer inattendu"

model = AutoModelForCausalLM.from_pretrained(BASE_ID, dtype=torch.float32, device_map="cpu")
emb_old = model.get_input_embeddings().weight.data.clone()
print("matrice d'origine :", tuple(emb_old.shape))
print("embeddings tied   :", model.config.tie_word_embeddings)

In [ ]:
NEW_SIZE = len(ext_tok)
model.resize_token_embeddings(NEW_SIZE)
emb = model.get_input_embeddings().weight.data

with torch.no_grad():
    for old_id, new_id in id_map.items():
        emb[new_id] = emb_old[old_id]
print(f"{len(id_map)} lignes spéciales déplacées")

test_old, test_new = list(id_map.items())[0]
print("vecteurs identiques :", torch.allclose(emb[test_new], emb_old[test_old]))

### Initialisation mean-pooling

Chaque nouveau token démarre à la **moyenne des embeddings de ses anciens sous-tokens**.
Exemple : `jëfandikoo` était découpé en `['Ġj','Ã«','f','and','ik','oo']` son embedding
hérite de la moyenne de ces six vecteurs. C'est un démarrage à chaud plutôt qu'un vecteur aléatoire.

In [ ]:
nouveaux = {t: i for t, i in ext_tok.get_vocab().items() if V <= i < V + N}
print("nouveaux tokens :", len(nouveaux))

sans_sous_tokens = 0
with torch.no_grad():
    for tok_str, new_id in nouveaux.items():
        texte = ext_tok.convert_tokens_to_string([tok_str])
        sub   = [s for s in base_tok.encode(texte, add_special_tokens=False) if s < V]
        if not sub:
            sans_sous_tokens += 1
            continue
        emb[new_id] = emb_old[sub].mean(dim=0)

print("tokens sans sous-tokens exploitables :", sans_sous_tokens)

### Piège n°3 — ne PAS recalibrer la norme

Le mean-pooling produit des embeddings de norme réduite (**ratio ≈ 0,5**) : la moyenne de *k*
vecteurs de directions indépendantes voit sa norme chuter d'environ √k.

**C'est correct, et il ne faut surtout pas « corriger » ce ratio.**

Les embeddings de SmolLM3 sont **tied** : la même matrice sert d'entrée *et* de projection
de sortie, où le logit vaut `h · e`. Ramener la norme à 1,0 double les logits des nouveaux
tokens et provoque leur sur-prédiction massive.

Mesuré sur ce modèle :

| | ratio ≈ 0,5 (brut) | ratio = 1,0 (recalibré) |
|---|---|---|
| BPB | **2,16** | 5,07 |
| Prédictions sur nouveaux tokens | **0,0 %** | 99,9 % |
| CE des tokens anciens | **6,52** | 21,59 (pire que le hasard) |

La norme réduite agit comme une **régularisation implicite** qui protège le softmax
de tokens non encore entraînés.

In [ ]:
n_anciens  = emb_old[:V].norm(dim=1).mean().item()
n_nouveaux = emb[V:V+N].norm(dim=1).mean().item()
ratio = n_nouveaux / n_anciens
print(f"norme moyenne — anciens : {n_anciens:.4f} | nouveaux : {n_nouveaux:.4f}")
print(f"ratio : {ratio:.3f}")
print("→ ~0,5 attendu et CORRECT (embeddings tied) — NE PAS recalibrer"
      if ratio < 0.7 else "→ ANOMALIE : un recalibrage a été appliqué, l'annuler")

### Remapper les identifiants spéciaux

Sans cette étape, `eos_token_id` pointerait vers un mot wolof et le modèle ne saurait plus
où s'arrêter de générer.

In [ ]:
def remap(val):
    if isinstance(val, list): return [id_map.get(v, v) for v in val]
    return id_map.get(val, val) if isinstance(val, int) else val

for cfg in [model.config, model.generation_config]:
    if cfg is None: continue
    for champ in ["eos_token_id", "bos_token_id", "pad_token_id", "decoder_start_token_id"]:
        ancien = getattr(cfg, champ, None)
        if ancien is not None:
            setattr(cfg, champ, remap(ancien))
            print(f"{champ}: {ancien} → {getattr(cfg, champ)}")

model.config.vocab_size = NEW_SIZE
print("vocab_size :", model.config.vocab_size)

In [ ]:
# Contrôles avant sauvegarde
assert model.get_input_embeddings().weight.shape[0] == len(ext_tok)

for champ in ["eos_token_id", "bos_token_id"]:
    tid = getattr(model.config, champ, None)
    if tid is not None:
        print(f"{champ}={tid} → {ext_tok.decode([tid] if isinstance(tid,int) else tid)!r}")

for m in ["jëfandikoo", "ñu", "dafay"]:
    ids = ext_tok.encode(" "+m, add_special_tokens=False)
    print(f"{m:12} {len(ids)} token(s) id={ids}")

In [ ]:
MODEL_DIR = f"{ROOT}/models/smollm3-wolof-ext{N_FINAL}-init"
os.makedirs(MODEL_DIR, exist_ok=True)
model.to(torch.bfloat16)
model.save_pretrained(MODEL_DIR, safe_serialization=True)
ext_tok.save_pretrained(MODEL_DIR)
print("modèle initialisé sauvegardé :", MODEL_DIR)

## Phase 6  BPB de référence

### Pourquoi le BPB et pas la perplexité

La perplexité se calcule **par token**. Le modèle étendu produit nettement moins de tokens
sur le même texte : il obtiendrait donc une perplexité différente **sans être meilleur ni pire**.

Le BPB normalise par les **octets du texte brut**, identiques dans les deux cas.
C'est la seule façon valide de comparer les configurations `défaut` et `étendu`.

```
BPB = somme(cross-entropy en nats) / (ln 2 × nombre d'octets UTF-8)
```

La perplexité reste utile pour comparer des runs partageant le **même** tokenizer.

In [ ]:
import math
from torch.nn import functional as F

@torch.no_grad()
def compute_bpb(model, tokenizer, textes, max_len=1024, device="cuda"):
    model.eval().to(device)
    total_nats, total_bytes, total_tokens = 0.0, 0, 0
    for t in textes:
        t = str(t)
        b = len(t.encode("utf-8"))
        if b == 0: continue
        ids = tokenizer.encode(t, add_special_tokens=False)[:max_len]
        if len(ids) < 2: continue
        x = torch.tensor([ids], device=device)
        logits = model(x).logits[0, :-1]
        cible  = x[0, 1:]
        total_nats  += F.cross_entropy(logits.float(), cible, reduction="sum").item()
        total_bytes += b
        total_tokens += len(ids) - 1
    return {"bpb": total_nats / (math.log(2) * total_bytes),
            "ppl": math.exp(total_nats / total_tokens),
            "n_bytes": total_bytes, "n_tokens": total_tokens}

In [ ]:
textes = eval_df["text"].astype(str).tolist()[:1000]

base_model = AutoModelForCausalLM.from_pretrained(BASE_ID, dtype=torch.bfloat16, device_map="cuda")
r_def = compute_bpb(base_model, base_tok, textes)
print(f"DÉFAUT   BPB={r_def['bpb']:.4f}  PPL={r_def['ppl']:.2f}  tokens={r_def['n_tokens']:,}")
del base_model; torch.cuda.empty_cache()

r_ext = compute_bpb(model, ext_tok, textes)
print(f"ÉTENDU   BPB={r_ext['bpb']:.4f}  PPL={r_ext['ppl']:.2f}  tokens={r_ext['n_tokens']:,}")

print(f"\noctets identiques : {r_def['n_bytes'] == r_ext['n_bytes']}   (doit être True)")
print(f"écart BPB : {r_ext['bpb'] - r_def['bpb']:+.4f}")

## Résumé des livrables

| Fichier | Contenu |
|---|---|
| `data/wolof_eval_v1.parquet` | split d'évaluation figé — **ne jamais modifier** |
| `data/wolof_bpe_train_v1.parquet` | split d'entraînement |
| `tokenizers/smollm3_wolof_ext_8000/` | tokenizer étendu + `extension_meta.json` |
| `models/smollm3-wolof-ext8000-init/` | modèle redimensionné et initialisé |

## Les trois pièges rencontrés

1. **`vocab_size` ≠ `len(tokenizer)`** → toujours dimensionner sur `len(tokenizer)`.
2. **Collision d'identifiants** → réindexer, en décalant les tokens spéciaux au-dessus des nouveaux.
3. **Recalibrage de la norme** → à ne PAS faire avec des embeddings *tied*.